In [2]:
import numpy as np
import rasterio
from scipy.signal import convolve2d
from scipy.ndimage import gaussian_filter

The below function `adv_synthetic_ndvi` and `simple_synthetic_ndvi` are temporary functions. Once the google earth engine is accessible by this project, it will be only used for short tests.

In [ ]:
def adv_synthetic_ndvi(shape = (300,300), seed = -1):
  """
  This function creates a randomly generated numpy array to simulate ndvi data
  from GEE, can pass a set size for the array and a set seed if it is needed
  """

  # only use seed if one was provided
  if (seed != -1):
    np.random.seed(seed)

  # create the first layer of noise
  noise = np.random.uniform(0,1, shape)


  # using a gaussian blur to smooth the noise for a more natural array
  smoothed = gaussian_filter(noise, sigma = 5)

  # normalizing back to farm and surrounding levels
  ndvi = 0.15 + (smoothed - smoothed.min()) / (smoothed.max() - \
    smoothed.min()) * 0.70

In [ ]:
def simple_synthetic_ndvi(shape=(100, 100)):
    """
    even more simple proof of concept of `adv_synthetic_ndvi`
    """
    # create empty numpy array
    ndvi = np.zeros(shape, dtype=np.float32)

    # locations of farms (high ndvi)
    ndvi[5:45, 5:45]   = 0.8
    ndvi[5:45, 55:95]  = 0.8
    ndvi[55:95, 5:45]  = 0.8
    ndvi[55:95, 55:95] = 0.8

    return ndvi

The below function `conv_kernal` is meant to create a different covolutional matrix for the sir model, one infected square can, by default, spread up to 3 grids away, with % decreasing in accordance to inverse square law. <br>
Currently this kernel is using aribtrary values. Before real life application, ideally each disease this cnn will be trained to detect has a dedicated CK for it and diseases with similar spreading capabilities. Future improvements could include wind or water changing the spread likelihood.

In [ ]:
def conv_kernal(radius = 3):
  """
  creates a ck with size 2*radius + 1
  this allows infections to spread up to 3 tiles away
  """
  size = 2 * radius + 1
  kernel = np.zeros((size,size))
  for y in range(size):
    for x in range(size):
      if x == radius and y == radius:
        kernel[y,x] = 0 # plants cannot reinfect themselves
        continue

Shape: (100, 100)
Unique values in array: [0.  0.8]
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [ ]:
def generate_wind_kernel(radius = 2, sigma=1.0, wind_x=0.0, wind_y=0.0):
    """
    convolutional matrix but with a wind vector

    inputs
    radius - distance a disease can spread
    sigma - spread of the disease through the wind
    wind_x - horizontal component of wind (0 blows east (right))
    wind_y - vertical component of wind (0 blows north (up))

    returns
    conv_matrix
    """

    # radius of 2 = 5x5 conv matrix
    size = 2*radius + 1


    # Create a coordinate grid centered at (0,0)
    # creating the array
    ax = np.linspace(-(size // 2), size // 2, size)
    xx, yy = np.meshgrid(ax, ax)

    # calculating the probabilities with the wind
    kernel = np.exp(-((xx - wind_x)**2 + (yy - wind_y)**2) / (2 * sigma**2))

    # return and normalize
    return kernel / np.sum(kernel)


Below is the final simulation code, what is changed above should not alter the core structure of the simulation step function.

In [ ]:
def sir_spatial_step(S, I, R, ndvi, conv_matrix, beta, gamma, dt=1.0):
    """
    single step in the simulation

    inputs

    S, I, R - numpy arrays of same shape
    ndvi - array of plant begetation density(influences spreadability)
    conv_matrix convulutional matrix for optimized disease spread
    beta - transmission rate
    gamma - Recovery/removal rate (not able to be reinfected)
    dt - time step size (default 1)

    returns
    S_next, I_next, R_next - next step of the simulation
    """
    # calculating base chance for tiles to be infected
    # same - do not change the output shape
    # fill - assumes nearby vegetation is not undergoing the same disease yet
    infected_pressure = convolve2d(I, conv_matrix, mode='same', boundary='fill')


    # adjusting transmission with beta, ndvi, and time using susceptibile array
    new_infections = beta * S * ndvi * infected_pressure * dt

    # calculating recoveries with gamma and dt
    new_recoveries = gamma * I * dt

    # making sure not to infect or recover more than what is available in a cell
    # ensures further calculations do not go negative
    new_infections = np.minimum(new_infections, S)
    new_recoveries = np.minimum(new_recoveries, I)

    # saving the new SIR numpy arrays
    S_next = S - new_infections
    I_next = I + new_infections - new_recoveries
    R_next = R + new_recoveries

    # realign values to our ranges
    S_next = np.clip(S_next, 0.0, 1.0)
    I_next = np.clip(I_next, 0.0, 1.0)
    R_next = np.clip(R_next, 0.0, 1.0)

    return S_next, I_next, R_next

Start simulation code.<br>
potential future improvements<br>
`CK` represents a strong wind in a compass direction<br>
Multiple random small diseases that start up and die<br>
saving each step to a file and labeling the type of disease<br>
using available information to estimate beta and gamma for real diseases

In [6]:
def random_simulation(shape = (100,100), timesteps = 100, beta = -1, gamma = -1):

  """
  Function to bulk run random simulations
  #TODO look into proper distrubtion of beta and gamma values for plant diseases
  currently incomplete

  inputs

  shape - size of the simulation
  timesteps - number of timesteps for the simulation before completion
  beta - infection rate (0 = cannot infect 1 = highly infectious)
  gamma - recovery/death rate (0 = cannot recover 1 = all recover instantly)

  returns
  nothing (will save each simulation step to file later)

  """

  if (beta == -1):
    #TODO random beta
    beta = .3
  if (gamma == -1):
    #TODO random gamma
    gamma = .05


  beta = 0.3   # infection rate (0 = cannot infect 1 = highly infectious)
  gamma = 0.05 # recovery/death rate

  for t in range(timesteps):
      S, I, R = sir_spatial_step(S, I, R, ndvi, conv_matrix, beta, gamma)

      # save history of simulation to train the cnn currently every 10 steps
      # while bugs are still being testind
      if t % 10 == 0:
          print(f"Step {t}: Total Infected Fraction = {np.mean(I):.4f}")